# DS2/GSE141383 — post-QC Scrublet doublet diagnostic

This notebook reproduces the original Scrublet check performed on the
complete post-QC DS2 object.

Interpretation:

- Scrublet is run after dataset-level QC.
- `doublet_score` and `predicted_doublet` are retained as diagnostic
  metadata.
- Predicted doublets are not removed.
- The analytical cell count therefore remains 21,391.

The original run predicted one doublet.

Input:
`data/processed/DS2_GSE141383/GSE141383_postQC.h5ad`

Output:
`data/processed/DS2_GSE141383/GSE141383_postQC_scrublet_checked.h5ad`

In [1]:
import inspect
import json
import os
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

print("Python:", sys.version)
print("scanpy:", sc.__version__)

Python: 3.12.3 | packaged by conda-forge | (main, Apr 15 2024, 18:20:11) [MSC v.1938 64 bit (AMD64)]
scanpy: 1.11.5


C:\Users\AmirSilco\AppData\Local\Temp\ipykernel_16888\2866864517.py:15: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print("scanpy:", sc.__version__)


In [2]:
def find_repository_root(start: Path) -> Path:
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start


PROJECT_ROOT = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", find_repository_root(Path.cwd()))
).expanduser().resolve()

INPUT_H5AD = Path(
    os.environ.get(
        "GSE141383_POSTQC_H5AD",
        PROJECT_ROOT
        / "data"
        / "processed"
        / "DS2_GSE141383"
        / "GSE141383_postQC.h5ad",
    )
).expanduser().resolve()

OUTPUT_H5AD = Path(
    os.environ.get(
        "GSE141383_SCRUBLET_H5AD",
        PROJECT_ROOT
        / "data"
        / "processed"
        / "DS2_GSE141383"
        / "GSE141383_postQC_scrublet_checked.h5ad",
    )
).expanduser().resolve()

AUDIT_DIR = (
    PROJECT_ROOT
    / "results"
    / "qc"
    / "DS2_GSE141383"
    / "00c_doublet_diagnostic"
)
FIGURE_DIR = AUDIT_DIR / "figures"

OUTPUT_H5AD.parent.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_INPUT_SHAPE = (21_391, 25_835)
EXPECTED_ORIGINAL_PREDICTED_DOUBLETS = 1

REQUESTED_SCRUBLET_PARAMETERS = {
    "sim_doublet_ratio": 2.0,
    "expected_doublet_rate": 0.05,
    "n_prin_comps": 30,
    "random_state": 0,
}

print("Input:", INPUT_H5AD)
print("Output:", OUTPUT_H5AD)

if not INPUT_H5AD.exists():
    raise FileNotFoundError(
        f"Post-QC input not found: {INPUT_H5AD}\n"
        "Run 00b_ds2_quality_control.ipynb first or set "
        "GSE141383_POSTQC_H5AD."
    )

Input: C:\Users\AmirSilco\Documents\GitHub\glioma-cnv-single-cell-atlas\data\processed\DS2_GSE141383\GSE141383_postQC.h5ad
Output: C:\Users\AmirSilco\Documents\GitHub\glioma-cnv-single-cell-atlas\data\processed\DS2_GSE141383\GSE141383_postQC_scrublet_checked.h5ad


In [3]:
def matrix_values(matrix):
    return matrix.data if sp.issparse(matrix) else np.asarray(matrix).ravel()


def assert_raw_counts(matrix, label: str):
    values = matrix_values(matrix)
    if values.size == 0:
        raise ValueError(f"{label} contains no nonzero count values.")
    if not np.isfinite(values).all():
        raise ValueError(f"{label} contains non-finite values.")
    if np.any(values < 0):
        raise ValueError(f"{label} contains negative values.")
    if not np.equal(values, np.floor(values)).all():
        raise ValueError(f"{label} is not an integer raw-count matrix.")

In [4]:
adata = sc.read_h5ad(INPUT_H5AD)

if adata.shape != EXPECTED_INPUT_SHAPE:
    raise AssertionError(
        f"Expected input shape {EXPECTED_INPUT_SHAPE}, observed {adata.shape}"
    )
if "sample" not in adata.obs:
    raise KeyError("adata.obs['sample'] is required.")
if "counts" not in adata.layers:
    raise KeyError("The post-QC object is missing layers['counts'].")

assert_raw_counts(adata.X, "adata.X")
assert_raw_counts(adata.layers["counts"], 'adata.layers["counts"]')

for column in ("doublet_score", "predicted_doublet"):
    if column in adata.obs:
        del adata.obs[column]

print(adata)

AnnData object with n_obs × n_vars = 21391 × 25835
    obs: 'cell_barcode', 'sample', 'sample_id', 'batch', 'geo_sample_id', 'geo_id', 'core_dataset', 'source_file', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb'
    var: 'gene_symbols', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'ensembl_id_version', 'n_cells'
    uns: 'build_timestamp', 'dataset_id', 'dataset_level_QC', 'geo_accession', 'input_format', 'source_files'
    layers: 'counts'


In [5]:
scrublet_signature = inspect.signature(sc.pp.scrublet)
supported_parameters = set(scrublet_signature.parameters)

scrublet_parameters = {
    key: value
    for key, value in REQUESTED_SCRUBLET_PARAMETERS.items()
    if key in supported_parameters
}

omitted_parameters = sorted(
    set(REQUESTED_SCRUBLET_PARAMETERS) - set(scrublet_parameters)
)
if omitted_parameters:
    print(
        "Parameters not supported by this Scanpy version and omitted:",
        omitted_parameters,
    )

print("Scrublet parameters used:", scrublet_parameters)

sc.pp.scrublet(
    adata,
    **scrublet_parameters,
)

required_columns = {"doublet_score", "predicted_doublet"}
missing = required_columns - set(adata.obs.columns)
if missing:
    raise RuntimeError(
        f"Scrublet completed without creating required columns: {missing}"
    )

adata.obs["predicted_doublet"] = (
    adata.obs["predicted_doublet"].astype(bool)
)

predicted_count = int(adata.obs["predicted_doublet"].sum())
print(adata.obs["predicted_doublet"].value_counts())
print(adata.obs["doublet_score"].describe())

if predicted_count != EXPECTED_ORIGINAL_PREDICTED_DOUBLETS:
    print(
        "WARNING: predicted-doublet count differs from the original run "
        f"({EXPECTED_ORIGINAL_PREDICTED_DOUBLETS}). Observed: "
        f"{predicted_count}. A package-version difference is a likely "
        "explanation."
    )

Scrublet parameters used: {'sim_doublet_ratio': 2.0, 'expected_doublet_rate': 0.05, 'n_prin_comps': 30, 'random_state': 0}
predicted_doublet
False    21390
True         1
Name: count, dtype: int64
count    21391.000000
mean         0.042284
std          0.040779
min          0.003048
25%          0.021286
50%          0.031973
75%          0.048732
max          0.742373
Name: doublet_score, dtype: float64


In [6]:
per_sample_summary = (
    adata.obs.groupby("sample", observed=True)
    .agg(
        cells=("predicted_doublet", "size"),
        predicted_doublets=("predicted_doublet", "sum"),
        mean_doublet_score=("doublet_score", "mean"),
        median_doublet_score=("doublet_score", "median"),
        maximum_doublet_score=("doublet_score", "max"),
    )
    .reset_index()
)
per_sample_summary["predicted_doublet_fraction"] = (
    per_sample_summary["predicted_doublets"]
    / per_sample_summary["cells"]
)
per_sample_summary.to_csv(
    AUDIT_DIR / "GSE141383_scrublet_summary_by_sample.tsv",
    sep="\t",
    index=False,
)

adata.obs[
    ["sample", "doublet_score", "predicted_doublet"]
].to_csv(
    AUDIT_DIR / "GSE141383_scrublet_per_cell_scores.tsv.gz",
    sep="\t",
    compression="gzip",
)

per_sample_summary

,sample,cells,predicted_doublets,mean_doublet_score,median_doublet_score,maximum_doublet_score,predicted_doublet_fraction
0,PDC001,6325,1,0.039842,0.030318,0.742373,0.000158
1,PJ052,1364,0,0.035920,0.028754,0.411992,0.000000
2,PJ053,3034,0,0.044416,0.033130,0.357022,0.000000
3,PW016-703,1530,0,0.052072,0.038251,0.532020,0.000000
4,PW017-703,1514,0,0.047594,0.033130,0.656627,0.000000
5,PW032-706,4746,0,0.038554,0.030859,0.588076,0.000000
6,PW032-710,979,0,0.048899,0.040404,0.485327,0.000000
7,PW035-710,953,0,0.036193,0.021669,0.532020,0.000000
8,PW039-705,946,0,0.054615,0.043505,0.656627,0.000000


In [7]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(
    adata.obs["doublet_score"].to_numpy(),
    bins=60,
    edgecolor="black",
)
ax.set_xlabel("Scrublet doublet score")
ax.set_ylabel("Cells")
ax.set_title("GSE141383 post-QC Scrublet scores")
fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "GSE141383_scrublet_score_histogram.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close(fig)

In [8]:
predicted_count = int(adata.obs["predicted_doublet"].sum())

doublet_summary = {
    "dataset_id": "DS2_GSE141383",
    "geo_accession": "GSE141383",
    "scope": "complete post-QC DS2 object",
    "method": "scanpy.pp.scrublet",
    "parameters_requested": REQUESTED_SCRUBLET_PARAMETERS,
    "parameters_used": scrublet_parameters,
    "input_cells": int(adata.n_obs),
    "input_genes": int(adata.n_vars),
    "predicted_doublets": predicted_count,
    "predicted_doublet_fraction": float(predicted_count / adata.n_obs),
    "doublets_removed": False,
    "output_cells": int(adata.n_obs),
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "output_h5ad": str(OUTPUT_H5AD),
}

adata.uns["doublet_detection_provenance"] = {
    "scope": "post_QC_dataset_level_diagnostic",
    "method": "scanpy.pp.scrublet",
    "doublets_removed": "false",
    "predicted_doublets": predicted_count,
    "parameters_json": json.dumps(scrublet_parameters, sort_keys=True),
}

with open(
    AUDIT_DIR / "GSE141383_scrublet_summary.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(doublet_summary, handle, indent=2)

print(doublet_summary)

{'dataset_id': 'DS2_GSE141383', 'geo_accession': 'GSE141383', 'scope': 'complete post-QC DS2 object', 'method': 'scanpy.pp.scrublet', 'parameters_requested': {'sim_doublet_ratio': 2.0, 'expected_doublet_rate': 0.05, 'n_prin_comps': 30, 'random_state': 0}, 'parameters_used': {'sim_doublet_ratio': 2.0, 'expected_doublet_rate': 0.05, 'n_prin_comps': 30, 'random_state': 0}, 'input_cells': 21391, 'input_genes': 25835, 'predicted_doublets': 1, 'predicted_doublet_fraction': 4.6748632602496374e-05, 'doublets_removed': False, 'output_cells': 21391, 'timestamp': '2026-08-03T16:20:39', 'output_h5ad': 'C:\\Users\\AmirSilco\\Documents\\GitHub\\glioma-cnv-single-cell-atlas\\data\\processed\\DS2_GSE141383\\GSE141383_postQC_scrublet_checked.h5ad'}


In [10]:
from pathlib import Path

import scanpy as sc


# ------------------------------------------------------------------
# Save the DS2 doublet-analysis output atomically
# ------------------------------------------------------------------
OUTPUT_H5AD = Path(OUTPUT_H5AD)
OUTPUT_H5AD.parent.mkdir(parents=True, exist_ok=True)

temporary_output = OUTPUT_H5AD.with_name(
    f"{OUTPUT_H5AD.stem}.temporary{OUTPUT_H5AD.suffix}"
)

# Remove any temporary file left by an interrupted run.
if temporary_output.exists():
    temporary_output.unlink()

adata.write_h5ad(
    temporary_output,
    compression="gzip",
)

if not temporary_output.exists():
    raise FileNotFoundError(
        f"Temporary output was not created: {temporary_output}"
    )

temporary_output.replace(OUTPUT_H5AD)

if not OUTPUT_H5AD.exists():
    raise FileNotFoundError(
        f"Final output was not created: {OUTPUT_H5AD}"
    )


# ------------------------------------------------------------------
# Reopen and validate without using a context manager
# ------------------------------------------------------------------
check = sc.read_h5ad(
    OUTPUT_H5AD,
    backed="r",
)

try:
    observed_shape = tuple(check.shape)
    expected_shape = tuple(EXPECTED_INPUT_SHAPE)

    if observed_shape != expected_shape:
        raise AssertionError(
            "Saved-file validation failed: "
            f"expected {expected_shape}, observed {observed_shape}"
        )

    if check.obs_names.has_duplicates:
        raise AssertionError(
            "Saved output contains duplicated cell identifiers."
        )

    if check.var_names.has_duplicates:
        raise AssertionError(
            "Saved output contains duplicated feature identifiers."
        )

    if check.n_obs == 0:
        raise AssertionError(
            "Saved output contains zero cells."
        )

    if check.n_vars == 0:
        raise AssertionError(
            "Saved output contains zero features."
        )

    # Validate Scrublet outputs when they are part of the in-memory object.
    scrublet_columns = [
        column
        for column in ("doublet_score", "predicted_doublet")
        if column in adata.obs.columns
    ]

    for column in scrublet_columns:
        if column not in check.obs.columns:
            raise AssertionError(
                f"Saved output is missing adata.obs[{column!r}]."
            )

    # Preserve and validate the raw-count layer when present.
    if "counts" in adata.layers and "counts" not in check.layers:
        raise AssertionError(
            "Saved output is missing layers['counts']."
        )

finally:
    check.file.close()


# ------------------------------------------------------------------
# Report completion
# ------------------------------------------------------------------
print("Saved and validated:", OUTPUT_H5AD)
print("Validated shape:", expected_shape)

if scrublet_columns:
    print(
        "Validated Scrublet columns:",
        ", ".join(scrublet_columns),
    )

if "counts" in adata.layers:
    print("Validated layer: counts")

Saved and validated: C:\Users\AmirSilco\Documents\GitHub\glioma-cnv-single-cell-atlas\data\processed\DS2_GSE141383\GSE141383_postQC_scrublet_checked.h5ad
Validated shape: (21391, 25835)
Validated Scrublet columns: doublet_score, predicted_doublet
Validated layer: counts
